# 15 — Hybrid Search (Vector + Keyword)

Combine BM25 keyword search with vector similarity using Reciprocal Rank Fusion.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Documents and Search Functions

In [ ]:
DOCUMENTS = [
    Document(page_content="The Python GIL (Global Interpreter Lock) is a mutex that protects access to Python objects, preventing multiple threads from executing Python bytecodes at once.", metadata={"id": 1}),
    Document(page_content="asyncio is Python's built-in library for writing concurrent code using the async/await syntax.", metadata={"id": 2}),
    Document(page_content="Threading in Python allows concurrent execution but is limited by the GIL for CPU-bound tasks.", metadata={"id": 3}),
    Document(page_content="FastAPI uses async/await natively and can handle thousands of concurrent connections.", metadata={"id": 4}),
    Document(page_content="The multiprocessing module spawns separate processes, each with its own GIL, enabling true parallel execution.", metadata={"id": 5}),
    Document(page_content="Celery is a distributed task queue for Python that supports scheduling, retries, and multiple message brokers.", metadata={"id": 6}),
    Document(page_content="Python 3.13 introduces an experimental free-threaded mode that removes the GIL entirely.", metadata={"id": 7}),
]

def bm25_search(query, documents, k=3):
    query_terms = set(query.lower().split())
    scored = [(doc, sum(doc.page_content.lower().split().count(t) for t in query_terms)) for doc in documents]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score in scored[:k] if score > 0]

def reciprocal_rank_fusion(results_lists, k=60):
    scores, doc_map = {}, {}
    for results in results_lists:
        for rank, doc in enumerate(results):
            doc_id = doc.page_content[:50]
            doc_map[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (rank + k)
    return [doc_map[did] for did, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

## Run Hybrid Search

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vectorstore = Chroma.from_documents(DOCUMENTS, OpenAIEmbeddings(model="text-embedding-3-small"))
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
prompt = ChatPromptTemplate.from_template("Answer using context:\n{context}\n\nQuestion: {question}\nAnswer:")

for query in ["How does the GIL affect Python threading?", "What are the options for async programming in Python?"]:
    print(f"Q: {query}")
    vector_results = vector_retriever.invoke(query)
    bm25_results = bm25_search(query, DOCUMENTS, k=3)
    fused = reciprocal_rank_fusion([vector_results, bm25_results])[:3]
    print(f"  Fused: {len(fused)} docs")
    context = "\n\n".join(doc.page_content for doc in fused)
    answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": query})
    print(f"  Answer: {answer}\n")

vectorstore.delete_collection()